# Etapa 02 — ¿Qué puede decir una red sobre una decisión de infraestructura?

Consulta el dossier. Predice, calcula, explica y juzga el alcance antes de registrar tu decisión.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
# Encuentra la raíz tanto desde la carpeta del notebook como desde la raíz del libro.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'sa_mise' / '__init__.py').exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from sa_mise import datos, modelos, expediente
plt.rcParams.update({'figure.figsize': (10, 4), 'axes.grid': True, 'grid.alpha': .2})

In [2]:
from sa_mise import sector
CASO='ejemplo_docente'
# Usa el mismo identificador de tu equipo en las ocho etapas.

## 1. Cinco barras para comparar métodos
La red es ficticia. Las inyecciones suman cero. Primero observamos conectividad y después flujos; las contingencias son independientes.

In [3]:
import networkx as nx
g,iny=modelos.red_docente()
display(pd.DataFrame({'grado':dict(g.degree()),'intermediacion':nx.betweenness_centrality(g)}))
display(modelos.flujo_dc(g,iny));base_cont=modelos.contingencias_dc(g,iny);display(base_cont)

,grado,intermediacion
A,2,0.000000
B,3,0.250000
C,3,0.250000
D,2,0.083333
E,2,0.083333


,origen,destino,flujo_MW,limite_MW,carga_pct
0,A,B,60.151515,65,92.540793
1,A,C,39.848485,65,61.305361
2,B,C,13.030303,40,32.575758
3,B,D,47.121212,55,85.674931
4,C,E,62.878788,65,96.736597
5,D,E,-7.878788,50,15.757576


,salida,conectado,max_carga_pct,alcance
0,A–B,True,153.846154,DC despacho fijo; no certifica N-1 integral
1,A–C,True,153.846154,DC despacho fijo; no certifica N-1 integral
2,B–C,True,95.151515,DC despacho fijo; no certifica N-1 integral
3,B–D,True,169.230769,DC despacho fijo; no certifica N-1 integral
4,C–E,True,200.000000,DC despacho fijo; no certifica N-1 integral
5,D–E,True,100.000000,DC despacho fijo; no certifica N-1 integral


Un grafo conectado puede presentar sobrecarga bajo estas inyecciones. Una contingencia DC no revisa tensión ni estabilidad. Identifica el caso limitante de la tabla antes de proponer una intervención.

## 2. Reactancia y límite: dos intervenciones diferentes
Primero modificamos reactancia A–C sin cambiar enlaces. Después ampliamos el rating docente B–D desde el caso original. No son diseños de obra real.

In [4]:
g2=g.copy();g2['A']['C']['x_pu']*=3
assert nx.betweenness_centrality(g)==nx.betweenness_centrality(g2)
display(modelos.flujo_dc(g2,iny))
g3=g.copy();g3['B']['D']['limite_MW']*=2
comparacion=base_cont[['salida','max_carga_pct']].merge(modelos.contingencias_dc(g3,iny)[['salida','max_carga_pct']],on='salida',suffixes=('_base','_rating_mayor'))
display(comparacion)

,origen,destino,flujo_MW,limite_MW,carga_pct
0,A,B,79.769231,65,122.721893
1,A,C,20.230769,65,31.124260
2,B,C,27.743590,40,69.358974
3,B,D,52.025641,55,94.592075
4,C,E,57.974359,65,89.191321
5,D,E,-2.974359,50,5.948718


,salida,max_carga_pct_base,max_carga_pct_rating_mayor
0,A–B,153.846154,153.846154
1,A–C,153.846154,153.846154
2,B–C,95.151515,88.717949
3,B–D,169.230769,169.230769
4,C–E,200.000000,124.786325
5,D–E,100.000000,96.581197


Modificar reactancia cambia la distribución física. Aumentar el límite sin cambiar reactancia cambia la utilización permitida, no el flujo base. Mira qué contingencias siguen siendo críticas: una mejora local no garantiza resolver todo. Para transferirlo al SIN necesitamos parámetros y escenarios reales.

## Registrar el avance del equipo
Sustituye el campo de decisión por tu razonamiento y conserva el nivel de evidencia. El identificador es el mismo durante todo el caso.

In [5]:
decision_estudiante='POR COMPLETAR'
expediente.registrar(CASO,'infraestructura','Contraste de topología, reactancia y límites en cinco barras ficticias; no resultados del STN','descripcion' if 2 in [0,1] else 'exploracion',
 'Extractos locales y/o supuestos identificados en DOSSIER y esta etapa',
 'Transformaciones, calendario y unidades explícitos en las celdas precedentes',
 'No constituye evaluación completa del SIN; distinguir cada resultado observado de los sintéticos',
 decision_estudiante)

Out[5]: 
{'version': 2,
 'caso': 'ejemplo_docente',
 'etapas': {'evidencia': {'hallazgo': 'Año 2024: generación 83262.923 GWh y pico 11704.365 MW; son magnitudes diferentes',
   'nivel': 'descripcion',
   'fuente': 'Extractos locales y/o supuestos identificados en DOSSIER y esta etapa',
   'transformacion': 'Transformaciones, calendario y unidades explícitos en las celdas precedentes',
   'limite': 'No constituye evaluación completa del SIN; distinguir cada resultado observado de los sintéticos',
   'decision_estudiante': 'POR COMPLETAR'},
  'identidad': {'hallazgo': "Asociaciones mensuales: {'meses': 264, 'correlacion_bruta': -0.3654022452929349, 'correlacion_residual': -0.5459860648178948, 'interpretacion': 'Asociación; controles aditivos no identifican causalidad ni corrigen toda no estacionariedad'}",
   'nivel': 'descripcion',
   'fuente': 'Extractos locales y/o supuestos identificados en DOSSIER y esta etapa',
   'transformacion': 'Transformaciones, calendario y unidades explícit